# Bài tập: Chương 5 – Fundamentals of Machine Learning / Nền tảng học máy

Sinh viên sẽ thực hành các nội dung:

1. **Generalization / Tổng quát hóa**
2. **Overfitting và underfitting / Học quá khớp và học thiếu**
3. **Noise features / Đặc trưng nhiễu**
4. **Random labels / Nhãn bị xáo trộn**
5. **Training set, validation set, test set**
6. **Learning rate / Tốc độ học**
7. **Model capacity / Năng lực mô hình**
8. **L2 regularization / Điều chuẩn L2**
9. **Dropout / Tắt ngẫu nhiên neuron khi huấn luyện**
10. **Early stopping / Dừng sớm**

## Yêu cầu nộp bài

- Chạy toàn bộ notebook.
- Hoàn thành các phần `TODO`.
- Trả lời câu hỏi phân tích sau mỗi bài.
- Lưu lại các biểu đồ loss/accuracy.

> Gợi ý: Khi so sánh model, không chỉ nhìn training accuracy. Hãy ưu tiên validation loss và validation accuracy.

## 0. Chuẩn bị môi trường

Cell này import các thư viện cần thiết và thiết lập seed để kết quả có tính lặp lại tương đối.

Các comment trong code được viết bằng tiếng Việt để sinh viên dễ theo dõi.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

# Đặt seed để kết quả có thể lặp lại tương đối.
# Lưu ý: nếu chạy trên GPU, kết quả vẫn có thể dao động nhẹ.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Cấu hình hiển thị biểu đồ.
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# Dùng subset nhỏ để notebook chạy nhanh trên máy sinh viên.
# Nếu máy mạnh, có thể tăng các giá trị này.
MNIST_SAMPLE_SIZE = 20000
IMDB_EPOCHS = 12
IMDB_BATCH_SIZE = 512

print("TensorFlow version:", tf.__version__)

## 1. Hàm tiện ích dùng chung

Các hàm này dùng để:

- Vẽ training loss và validation loss.
- Vẽ training accuracy và validation accuracy.
- So sánh nhiều model trên cùng một biểu đồ.

Sinh viên không cần sửa cell này.

In [ ]:
def plot_train_val(history, title_prefix="Model"):
    """
    Vẽ training loss/validation loss và training accuracy/validation accuracy.
    Dùng để quan sát underfitting, overfitting và generalization.
    """
    epochs = range(1, len(history.history["loss"]) + 1)

    plt.figure()
    plt.plot(epochs, history.history["loss"], "o-", label="Training loss")
    plt.plot(epochs, history.history["val_loss"], "o-", label="Validation loss")
    plt.title(f"{title_prefix}: Training loss vs Validation loss")
    plt.xlabel("Epochs / Số epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    if "accuracy" in history.history:
        plt.figure()
        plt.plot(epochs, history.history["accuracy"], "o-", label="Training accuracy")
        plt.plot(epochs, history.history["val_accuracy"], "o-", label="Validation accuracy")
        plt.title(f"{title_prefix}: Training accuracy vs Validation accuracy")
        plt.xlabel("Epochs / Số epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.show()


def plot_compare_histories(histories, metric, title, ylabel=None):
    """
    So sánh cùng một metric của nhiều model.

    histories: dict dạng {"Tên model": history}
    metric: ví dụ "val_loss" hoặc "val_accuracy"
    title: tiêu đề biểu đồ
    """
    plt.figure()
    for name, history in histories.items():
        values = history.history[metric]
        epochs = range(1, len(values) + 1)
        plt.plot(epochs, values, marker="o", label=name)

    plt.title(title)
    plt.xlabel("Epochs / Số epoch")
    plt.ylabel(ylabel if ylabel is not None else metric)
    plt.legend()
    plt.show()


def print_best_validation_result(history, metric="val_loss", mode="min"):
    """
    In ra epoch tốt nhất theo validation metric.

    mode="min": dùng cho val_loss
    mode="max": dùng cho val_accuracy
    """
    values = history.history[metric]
    if mode == "min":
        best_index = int(np.argmin(values))
    else:
        best_index = int(np.argmax(values))

    print(f"Metric: {metric}")
    print(f"Epoch tốt nhất: {best_index + 1}")
    print(f"Giá trị tốt nhất: {values[best_index]:.4f}")

# Bài tập 1 – Noise features và generalization

## Mục tiêu

Hiểu vì sao thêm đặc trưng nhiễu có thể làm model dễ overfit và generalize kém hơn.

Trong bài này, ta dùng MNIST:

- Mỗi ảnh gốc có 784 pixel.
- Ta tạo thêm 784 chiều nhiễu ngẫu nhiên.
- Ta cũng tạo thêm 784 chiều toàn số 0 để so sánh.

Hai dataset sau khi nối đều có shape `(số_mẫu, 1568)`, nhưng dataset có noise channels có thể gây khó khăn hơn cho model.

In [ ]:
from tensorflow.keras.datasets import mnist

# Tải dữ liệu MNIST.
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Chuyển ảnh 28x28 thành vector 784 chiều và chuẩn hóa về [0, 1].
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255.0
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255.0

# Dùng một phần dữ liệu để chạy nhanh.
train_images_demo = train_images[:MNIST_SAMPLE_SIZE]
train_labels_demo = train_labels[:MNIST_SAMPLE_SIZE]

print("Shape train_images_demo:", train_images_demo.shape)
print("Shape train_labels_demo:", train_labels_demo.shape)

### Nhiệm vụ 1.1

Hoàn thành code để tạo:

- `train_images_with_noise_channels`
- `train_images_with_zeros_channels`

Gợi ý:

- Dùng `np.concatenate([...], axis=1)` để nối theo chiều feature/cột.
- Noise có shape `(len(train_images_demo), 784)`.
- Zero channels cũng có shape `(len(train_images_demo), 784)`.

In [ ]:
# TODO 1.1: Tạo 784 chiều nhiễu ngẫu nhiên.
noise_channels = np.random.random((len(train_images_demo), 784)).astype("float32")

# TODO 1.2: Tạo 784 chiều toàn số 0.
zero_channels = np.zeros((len(train_images_demo), 784), dtype="float32")

# TODO 1.3: Nối dữ liệu gốc với noise_channels theo axis=1.
train_images_with_noise_channels = np.concatenate(
    [train_images_demo, noise_channels],
    axis=1
)

# TODO 1.4: Nối dữ liệu gốc với zero_channels theo axis=1.
train_images_with_zeros_channels = np.concatenate(
    [train_images_demo, zero_channels],
    axis=1
)

print("Shape dữ liệu có noise channels:", train_images_with_noise_channels.shape)
print("Shape dữ liệu có zero channels:", train_images_with_zeros_channels.shape)

### Nhiệm vụ 1.2

Hoàn thành hàm tạo model MNIST.

Yêu cầu:

- Input shape phụ thuộc vào `input_dim`.
- Hidden layer: `Dense(512, activation="relu")`
- Output layer: `Dense(10, activation="softmax")`
- Loss: `sparse_categorical_crossentropy`
- Optimizer: `rmsprop`
- Metric: `accuracy`

In [ ]:
def get_mnist_model(input_dim):
    """
    Tạo model Dense đơn giản cho bài toán MNIST.
    input_dim là số chiều input.
    """
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="rmsprop",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Kiểm tra nhanh model.
demo_model = get_mnist_model(input_dim=1568)
demo_model.summary()

### Nhiệm vụ 1.3

Train cùng một model trên 2 dataset:

- Dataset có noise channels.
- Dataset có zero channels.

Sau đó vẽ biểu đồ so sánh `val_accuracy`.

Câu hỏi phân tích:

1. Dataset nào có validation accuracy tốt hơn?
2. Vì sao noise channels có thể làm model generalize kém hơn?
3. Đây có phải là ví dụ của spurious correlations không? Vì sao?

In [ ]:
# Train model với noise channels.
model_noise = get_mnist_model(input_dim=train_images_with_noise_channels.shape[1])
history_noise = model_noise.fit(
    train_images_with_noise_channels,
    train_labels_demo,
    epochs=8,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# Train model với zero channels.
model_zero = get_mnist_model(input_dim=train_images_with_zeros_channels.shape[1])
history_zero = model_zero.fit(
    train_images_with_zeros_channels,
    train_labels_demo,
    epochs=8,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

# Vẽ biểu đồ so sánh validation accuracy.
plot_compare_histories(
    {
        "Noise channels": history_noise,
        "Zero channels": history_zero
    },
    metric="val_accuracy",
    title="Bài tập 1: Ảnh hưởng của noise features đến validation accuracy",
    ylabel="Validation accuracy"
)

# Bài tập 2 – Random labels và khả năng ghi nhớ của model

## Mục tiêu

Quan sát rằng model có thể fit training data ngay cả khi label bị xáo trộn ngẫu nhiên.

Điều này giúp phân biệt:

- **Optimization**: model giảm training loss.
- **Generalization**: model hoạt động tốt trên dữ liệu mới.

Nếu label bị xáo trộn, không còn quan hệ thật giữa ảnh và nhãn. Vì vậy validation accuracy không thể tốt một cách bền vững.

### Trả lời bài 1
1. Dataset có noise channels thường có validation accuracy thấp hơn do model dễ học theo nhiễu.
2. Noise features làm tăng nguy cơ overfitting vì model cố ghi nhớ thông tin không hữu ích.
3. Zero channels không ảnh hưởng nhiều vì chúng không mang thêm thông tin.

### Nhiệm vụ 2.1

Tạo bản sao của `train_labels_demo`, sau đó dùng `np.random.shuffle()` để xáo trộn label.

Sau đó train model trên ảnh thật nhưng label bị xáo trộn.

Câu hỏi phân tích:

1. Training accuracy có tăng không?
2. Validation accuracy có tăng tốt không?
3. Điều này nói gì về khả năng ghi nhớ của neural network?

In [ ]:
# TODO 2.1: Tạo label ngẫu nhiên bằng cách xáo trộn nhãn gốc.
random_train_labels = train_labels_demo.copy()
np.random.shuffle(random_train_labels)

# Train model với nhãn bị xáo trộn.
model_random_labels = get_mnist_model(input_dim=784)
history_random_labels = model_random_labels.fit(
    train_images_demo,
    random_train_labels,
    epochs=12,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

plot_train_val(history_random_labels, title_prefix="Bài tập 2: MNIST với random labels")

# Bài tập 3 – Training, validation và test set

## Mục tiêu

Hiểu vai trò của ba tập dữ liệu:

- **Training set**: dùng để học weights/bias.
- **Validation set**: dùng để chọn model hoặc hyperparameters.
- **Test set**: dùng để đánh giá cuối cùng.

Trong bài này, ta dùng IMDB review classification.

### Trả lời bài 2
1. Training accuracy vẫn có thể tăng dù label bị xáo trộn vì neural network có khả năng ghi nhớ dữ liệu.
2. Validation accuracy thường gần mức đoán ngẫu nhiên (~10% cho MNIST).
3. Điều này cho thấy optimization khác với generalization.

In [ ]:
from tensorflow.keras.datasets import imdb

# Chỉ lấy 10.000 từ phổ biến nhất.
(raw_train_data, raw_train_labels), (raw_test_data, raw_test_labels) = imdb.load_data(num_words=10000)

def vectorize_sequences(sequences, dimension=10000):
    """
    Chuyển mỗi review từ danh sách word index sang vector multi-hot.
    Ví dụ review có các word index [3, 10, 100]
    thì vector tại vị trí 3, 10, 100 sẽ bằng 1.
    """
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

# Vector hóa dữ liệu train/test.
x_all_train = vectorize_sequences(raw_train_data)
y_all_train = np.asarray(raw_train_labels).astype("float32")
x_test = vectorize_sequences(raw_test_data)
y_test = np.asarray(raw_test_labels).astype("float32")

print("x_all_train shape:", x_all_train.shape)
print("x_test shape:", x_test.shape)

### Nhiệm vụ 3.1

Tự tách dữ liệu training ban đầu thành:

- `x_train`, `y_train`: dùng để train model.
- `x_val`, `y_val`: dùng làm validation set.

Gợi ý:

- Dùng 10.000 mẫu đầu làm validation.
- Phần còn lại làm training.

In [ ]:
# TODO 3.1: Chia validation set và training set.
x_val = x_all_train[:10000]
y_val = y_all_train[:10000]

x_train = x_all_train[10000:]
y_train = y_all_train[10000:]

print("x_train:", x_train.shape)
print("x_val:", x_val.shape)
print("x_test:", x_test.shape)

### Nhiệm vụ 3.2

Tạo model IMDB cơ bản và đánh giá trên test set.

Câu hỏi phân tích:

1. Validation accuracy tốt nhất đạt ở epoch nào?
2. Test accuracy cuối cùng là bao nhiêu?
3. Vì sao không nên dùng test set để chọn số epoch hoặc chọn model?

### Trả lời bài 3
1. Validation accuracy tốt nhất thường xuất hiện trước khi model bắt đầu overfit.
2. Test accuracy thường thấp hơn một chút so với validation accuracy.
3. Không nên dùng test set để tuning hyperparameter vì sẽ gây rò rỉ thông tin.

In [ ]:
def build_imdb_model(units=16, regularizer=None, dropout_rate=None):
    """
    Tạo model phân loại review IMDB.

    units: số neuron trong mỗi hidden layer.
    regularizer: ví dụ regularizers.l2(0.002).
    dropout_rate: nếu khác None, thêm Dropout sau hidden layer.
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(10000,)))
    model.add(layers.Dense(units, activation="relu", kernel_regularizer=regularizer))

    if dropout_rate is not None:
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(units, activation="relu", kernel_regularizer=regularizer))

    if dropout_rate is not None:
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


model_basic = build_imdb_model(units=16)
history_basic = model_basic.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

plot_train_val(history_basic, title_prefix="Bài tập 3: IMDB basic model")
print_best_validation_result(history_basic, metric="val_accuracy", mode="max")

# Đánh giá cuối cùng trên test set.
test_loss, test_acc = model_basic.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Bài tập 4 – Beating a common-sense baseline

## Mục tiêu

Trước khi nói model tốt, model phải vượt qua một baseline đơn giản.

Trong bài toán binary classification IMDB:

- Baseline đơn giản có thể là luôn dự đoán class phổ biến nhất.
- Nếu dataset cân bằng, baseline này thường khoảng 50% accuracy.

Nhiệm vụ của sinh viên là tính baseline accuracy và so sánh với neural network.

In [ ]:
# TODO 4.1: Tính tỷ lệ positive và negative trong training set.
positive_ratio = np.mean(y_train)
negative_ratio = 1.0 - positive_ratio

print(f"Tỷ lệ positive review: {positive_ratio:.4f}")
print(f"Tỷ lệ negative review: {negative_ratio:.4f}")

# TODO 4.2: Baseline accuracy khi luôn dự đoán class phổ biến nhất.
common_sense_baseline_acc = max(positive_ratio, negative_ratio)
print(f"Common-sense baseline accuracy: {common_sense_baseline_acc:.4f}")

# TODO 4.3: So sánh baseline với validation accuracy tốt nhất của model_basic.
best_val_acc = max(history_basic.history["val_accuracy"])
print(f"Best validation accuracy của neural network: {best_val_acc:.4f}")
print(f"Chênh lệch so với baseline: {best_val_acc - common_sense_baseline_acc:.4f}")

### Câu hỏi phân tích bài 4

1. Baseline accuracy là bao nhiêu?
2. Neural network có vượt baseline không?
3. Nếu model không vượt baseline, ta nên kiểm tra những gì?
4. Vì sao baseline giúp ta tránh xây model phức tạp nhưng vô ích?

### Trả lời bài 4
1. Baseline accuracy là tỷ lệ của class xuất hiện nhiều nhất.
2. Neural network cần vượt baseline để chứng minh có học được thông tin hữu ích.
3. Nếu không vượt baseline, cần kiểm tra dữ liệu, preprocessing và model architecture.
4. Baseline giúp tránh đánh giá sai hiệu quả model.

# Bài tập 5 – Tuning learning rate

## Mục tiêu

Quan sát ảnh hưởng của learning rate đến quá trình training.

Learning rate quá lớn có thể làm model cập nhật weights quá mạnh, khiến training không ổn định.

Nhiệm vụ:

- Train cùng một model MNIST với nhiều learning rate.
- So sánh validation accuracy và validation loss.
- Rút ra learning rate nào phù hợp hơn.

In [ ]:
def train_mnist_with_learning_rate(learning_rate, epochs=8):
    """
    Train model MNIST với learning rate cho trước.
    """
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        train_images_demo,
        train_labels_demo,
        epochs=epochs,
        batch_size=128,
        validation_split=0.2,
        verbose=1
    )
    return history


# TODO 5.1: Thử ít nhất 3 learning rate khác nhau.
history_lr_1 = train_mnist_with_learning_rate(learning_rate=1.0, epochs=8)
history_lr_2 = train_mnist_with_learning_rate(learning_rate=1e-2, epochs=8)
history_lr_3 = train_mnist_with_learning_rate(learning_rate=1e-3, epochs=8)

plot_compare_histories(
    {
        "lr = 1.0": history_lr_1,
        "lr = 1e-2": history_lr_2,
        "lr = 1e-3": history_lr_3
    },
    metric="val_accuracy",
    title="Bài tập 5: So sánh validation accuracy theo learning rate",
    ylabel="Validation accuracy"
)

plot_compare_histories(
    {
        "lr = 1.0": history_lr_1,
        "lr = 1e-2": history_lr_2,
        "lr = 1e-3": history_lr_3
    },
    metric="val_loss",
    title="Bài tập 5: So sánh validation loss theo learning rate",
    ylabel="Validation loss"
)

### Câu hỏi phân tích bài 5

1. Learning rate nào cho validation accuracy tốt nhất?
2. Learning rate quá cao có biểu hiện gì trên loss/accuracy?
3. Learning rate quá thấp có thể gây vấn đề gì?
4. Vì sao learning rate là hyperparameter quan trọng?

### Trả lời bài 5
1. Learning rate khoảng 1e-3 hoặc 1e-2 thường ổn định hơn.
2. Learning rate quá cao làm loss dao động mạnh hoặc diverge.
3. Learning rate quá thấp khiến training rất chậm.
4. Learning rate ảnh hưởng trực tiếp đến tốc độ hội tụ.

# Bài tập 6 – Model capacity: model nhỏ, vừa và lớn

## Mục tiêu

Hiểu ảnh hưởng của model capacity đến underfitting và overfitting.

Ta so sánh 3 model IMDB:

- **Small model**: 4 units/layer
- **Original model**: 16 units/layer
- **Large model**: 512 units/layer

Sinh viên cần quan sát validation loss để nhận xét model nào underfit, model nào overfit mạnh hơn.

In [ ]:
# Train small model.
model_small = build_imdb_model(units=4)
history_small = model_small.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

# Train original model.
model_original = build_imdb_model(units=16)
history_original = model_original.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

# Train large model.
model_large = build_imdb_model(units=512)
history_large = model_large.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

plot_compare_histories(
    {
        "Small model - 4 units": history_small,
        "Original model - 16 units": history_original,
        "Large model - 512 units": history_large
    },
    metric="val_loss",
    title="Bài tập 6: Ảnh hưởng của model capacity đến validation loss",
    ylabel="Validation loss"
)

plot_compare_histories(
    {
        "Small model - 4 units": history_small,
        "Original model - 16 units": history_original,
        "Large model - 512 units": history_large
    },
    metric="val_accuracy",
    title="Bài tập 6: Ảnh hưởng của model capacity đến validation accuracy",
    ylabel="Validation accuracy"
)

### Câu hỏi phân tích bài 6

1. Model nào có validation loss thấp nhất?
2. Model lớn nhất có luôn tốt nhất không?
3. Dấu hiệu nào cho thấy model đang overfit?
4. Dấu hiệu nào cho thấy model có thể underfit?
5. Theo em, model nào có capacity vừa đủ nhất trong 3 model trên?

### Trả lời bài 6
1. Model vừa phải thường có validation loss thấp nhất.
2. Model lớn nhất không phải lúc nào cũng tốt nhất.
3. Overfitting thể hiện qua training accuracy tăng nhưng validation accuracy giảm.
4. Underfitting xảy ra khi cả training và validation accuracy đều thấp.

# Bài tập 7 – L2 regularization

## Mục tiêu

Hiểu cách L2 regularization giúp giảm overfitting bằng cách phạt weights quá lớn.

Công thức trực giác:

`Loss mới = Loss gốc + λ × tổng(w²)`

Nhiệm vụ:

- Train model gốc không có regularization.
- Train model có L2 regularization.
- So sánh validation loss.

In [ ]:
# Model không dùng regularization.
model_no_reg = build_imdb_model(units=16)
history_no_reg = model_no_reg.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

# TODO 7.1: Thêm L2 regularization.
# Có thể thử các giá trị: 0.001, 0.002, 0.01
model_l2 = build_imdb_model(
    units=16,
    regularizer=regularizers.l2(0.002)
)
history_l2 = model_l2.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

plot_compare_histories(
    {
        "No regularization": history_no_reg,
        "L2 regularization": history_l2
    },
    metric="val_loss",
    title="Bài tập 7: Ảnh hưởng của L2 regularization đến validation loss",
    ylabel="Validation loss"
)

plot_compare_histories(
    {
        "No regularization": history_no_reg,
        "L2 regularization": history_l2
    },
    metric="val_accuracy",
    title="Bài tập 7: Ảnh hưởng của L2 regularization đến validation accuracy",
    ylabel="Validation accuracy"
)

### Câu hỏi phân tích bài 7

1. Model có L2 có overfit chậm hơn không?
2. Validation loss của model L2 có ổn định hơn không?
3. Vì sao L2 phạt weights lớn có thể giúp model generalize tốt hơn?
4. Nếu hệ số L2 quá lớn, chuyện gì có thể xảy ra?

### Trả lời bài 7
1. Model có L2 thường overfit chậm hơn.
2. Validation loss ổn định hơn khi dùng regularization.
3. L2 giúp weights không quá lớn nên model generalize tốt hơn.
4. Nếu lambda quá lớn, model có thể underfit.

# Bài tập 8 – Dropout

## Mục tiêu

Hiểu Dropout là kỹ thuật regularization bằng cách tắt ngẫu nhiên một phần activation trong training.

Nhiệm vụ:

- Train model gốc.
- Train model có Dropout.
- So sánh validation loss.

Gợi ý:

- Thử `dropout_rate=0.5`.
- Có thể thử thêm `0.2` hoặc `0.3` nếu còn thời gian.

In [ ]:
# Model gốc.
model_no_dropout = build_imdb_model(units=16)
history_no_dropout = model_no_dropout.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

# TODO 8.1: Tạo model có Dropout.
model_dropout = build_imdb_model(units=16, dropout_rate=0.5)
history_dropout = model_dropout.fit(
    x_train,
    y_train,
    epochs=IMDB_EPOCHS,
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    verbose=1
)

plot_compare_histories(
    {
        "No Dropout": history_no_dropout,
        "Dropout 0.5": history_dropout
    },
    metric="val_loss",
    title="Bài tập 8: Ảnh hưởng của Dropout đến validation loss",
    ylabel="Validation loss"
)

plot_compare_histories(
    {
        "No Dropout": history_no_dropout,
        "Dropout 0.5": history_dropout
    },
    metric="val_accuracy",
    title="Bài tập 8: Ảnh hưởng của Dropout đến validation accuracy",
    ylabel="Validation accuracy"
)

### Câu hỏi phân tích bài 8

1. Dropout có làm validation loss tăng chậm hơn không?
2. Vì sao Dropout giúp model không phụ thuộc quá nhiều vào một vài neuron?
3. Dropout có được áp dụng ở test/inference time không?
4. Nếu dropout rate quá cao, model có thể gặp vấn đề gì?

### Trả lời bài 8
1. Dropout thường làm validation loss tăng chậm hơn.
2. Dropout giúp model không phụ thuộc quá nhiều vào một neuron cụ thể.
3. Dropout chỉ hoạt động khi training, không dùng ở inference.
4. Dropout là kỹ thuật regularization hiệu quả cho deep learning.

# Bài tập 9 – Early stopping

## Mục tiêu

Hiểu early stopping là kỹ thuật dừng training khi validation performance không còn cải thiện.

Trong thực tế, ta thường không muốn train đến epoch cuối nếu validation loss đã bắt đầu tăng.

Nhiệm vụ:

- Train model với số epoch lớn.
- Dùng `EarlyStopping` để tự động dừng sớm.
- So sánh số epoch thực tế đã train với số epoch tối đa.

In [ ]:
# TODO 9.1: Tạo callback EarlyStopping.
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",          # Theo dõi validation loss
    patience=3,                  # Chờ 3 epoch không cải thiện rồi dừng
    restore_best_weights=True    # Khôi phục weights tốt nhất
)

model_early = build_imdb_model(units=16)
history_early = model_early.fit(
    x_train,
    y_train,
    epochs=50,                   # Đặt lớn để callback có cơ hội dừng sớm
    batch_size=IMDB_BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

plot_train_val(history_early, title_prefix="Bài tập 9: EarlyStopping")
print("Số epoch thực tế đã train:", len(history_early.history["loss"]))
print_best_validation_result(history_early, metric="val_loss", mode="min")

# Đánh giá model early stopping trên test set.
test_loss_early, test_acc_early = model_early.evaluate(x_test, y_test, verbose=0)
print(f"Test loss với EarlyStopping: {test_loss_early:.4f}")
print(f"Test accuracy với EarlyStopping: {test_acc_early:.4f}")

### Câu hỏi phân tích bài 9

1. Model dừng ở epoch bao nhiêu?
2. Epoch có validation loss tốt nhất là epoch nào?
3. Vì sao `restore_best_weights=True` hữu ích?
4. Early stopping khác gì với việc train cố định 20 epoch?

### Trả lời bài 9
1. Model dừng khi validation loss không còn cải thiện.
2. Epoch tốt nhất là epoch có validation loss nhỏ nhất.
3. restore_best_weights giúp giữ lại model tốt nhất.
4. Early stopping giúp tránh overfitting và tiết kiệm thời gian train.

# Bài tập 10 – Tổng hợp: chọn model tốt nhất

## Mục tiêu

Sinh viên tổng hợp kết quả từ các bài trước để chọn model phù hợp nhất.

Hãy so sánh các model IMDB đã train:

- Basic model
- Small model
- Large model
- L2 model
- Dropout model
- EarlyStopping model

Tiêu chí so sánh:

- Validation loss thấp nhất.
- Validation accuracy cao nhất.
- Khoảng cách giữa training và validation có quá lớn không.
- Model có overfit nhanh không.
- Model có quá phức tạp không.

In [ ]:
# TODO 10.1: Tạo bảng tổng hợp kết quả các model.
# Lưu ý: một số history có thể chưa tồn tại nếu bạn chưa chạy các bài trước.

model_histories = {
    "Basic": history_basic,
    "Small": history_small,
    "Original": history_original,
    "Large": history_large,
    "L2": history_l2,
    "Dropout": history_dropout,
    "EarlyStopping": history_early
}

summary_rows = []
for name, hist in model_histories.items():
    best_val_loss = float(np.min(hist.history["val_loss"]))
    best_val_acc = float(np.max(hist.history["val_accuracy"]))
    best_epoch_loss = int(np.argmin(hist.history["val_loss"])) + 1
    best_epoch_acc = int(np.argmax(hist.history["val_accuracy"])) + 1
    total_epochs = len(hist.history["loss"])

    summary_rows.append([
        name,
        best_val_loss,
        best_epoch_loss,
        best_val_acc,
        best_epoch_acc,
        total_epochs
    ])

# In bảng bằng format đơn giản, không cần pandas.
print("Model | Best val_loss | Epoch loss tốt nhất | Best val_acc | Epoch acc tốt nhất | Tổng epochs")
print("-" * 95)
for row in summary_rows:
    print(f"{row[0]:15s} | {row[1]:.4f} | {row[2]:17d} | {row[3]:.4f} | {row[4]:16d} | {row[5]:11d}")

### Câu hỏi tổng hợp cuối bài

Trả lời ngắn gọn các câu hỏi sau:

1. Model nào có validation loss thấp nhất?
2. Model nào có validation accuracy cao nhất?
3. Model nào overfit nhanh nhất? Dựa vào biểu đồ nào?
4. L2 regularization có cải thiện generalization không?
5. Dropout có cải thiện generalization không?
6. Early stopping giúp tránh overfitting như thế nào?
7. Vì sao không nên chọn model chỉ dựa trên training accuracy?
8. Theo em, quy trình chọn model hợp lý nên gồm các bước nào?
9. Cách chọn được model với capacity vừa đủ?

### Trả lời tổng hợp
1. Model có validation loss thấp nhất thường là model dùng regularization hoặc early stopping.
2. Validation accuracy cao nhất thường thuộc về model cân bằng tốt giữa bias và variance.
3. Model lớn nhất thường overfit nhanh nhất.
4. Model tốt nhất không chỉ dựa vào training accuracy mà còn dựa vào khả năng generalization.

## Kết luận

Một model tốt không phải là model có training accuracy cao nhất, mà là model có khả năng **generalization / tổng quát hóa** tốt nhất trên dữ liệu chưa từng thấy.